In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.utils import to_categorical


In [2]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# Select only classes 0–4
train_mask = y_train < 5
test_mask = y_test < 5

X_train = X_train[train_mask][:2000]
y_train = y_train[train_mask][:2000]

X_test = X_test[test_mask][:500]
y_test = y_test[test_mask][:500]


In [4]:
X_train = tf.expand_dims(X_train, axis=-1)
X_test = tf.expand_dims(X_test, axis=-1)

X_train = tf.image.grayscale_to_rgb(X_train)
X_test = tf.image.grayscale_to_rgb(X_test)


In [5]:
X_train = tf.image.resize(X_train, (224, 224))
X_test = tf.image.resize(X_test, (224, 224))


In [6]:
X_train = X_train / 255.0
X_test = X_test / 255.0


In [7]:
y_train = to_categorical(y_train, 5)
y_test = to_categorical(y_test, 5)


In [8]:
base_model = MobileNet(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
base_model.trainable = False


In [10]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(64, activation='relu'),
    Dense(5, activation='softmax')
])


In [11]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [12]:
model.fit(X_train, y_train, epochs=5, batch_size=32)


Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 61s 881ms/step - accuracy: 0.6524 - loss: 0.8841
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 78s 836ms/step - accuracy: 0.8890 - loss: 0.3057
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 52s 830ms/step - accuracy: 0.9297 - loss: 0.2205
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 81s 822ms/step - accuracy: 0.9489 - loss: 0.1612
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 51s 812ms/step - accuracy: 0.9439 - loss: 0.1569


In [13]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_accuracy)


16/16 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.9043 - loss: 0.2548
Test Accuracy: 0.8999999761581421
